# MERRA-2 AOD vs AERONET-AOD @ Nghia Do — NE-monsoon dry season

**Question:** Does MERRA-2 (TOTEXTTAU, total aerosol extinction AOT @ 550 nm)
reproduce AERONET-observed AOD at Nghia Do (urban Hanoi) during the Nov-Mar
dry season?

Companion to the CAMS check — both are candidate priors. The Stage B note in
memory says Stage B uses CAMS only, but if MERRA-2 happens to be a better
matcher during haze, that's a meaningful design signal.

Comparisons are raw — no bias correction, no smoothing.


In [ ]:
import glob
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr

# Nghia Do — AERONET site in urban Hanoi
LAT, LON = 21.048, 105.800

# NE-monsoon dry season for northern Vietnam
DRY_MONTHS = [11, 12, 1, 2, 3]

# Window covered by MERRA-2 download + AERONET overlap (~2 dry seasons)
DATE_START = "2022-09-01"
DATE_END   = "2026-04-30"

AERONET_CSV = "/home/slow_data/Air_Quality/AERONET/process/aod_550/10/NGHIA_DO.csv"
MERRA2_ROOT = "/home/slow_data/Air_Quality/MERRA2/M2T1NXAER"


In [ ]:
def load_aeronet(csv_path):
    """AERONET file is UTC+7 (Hanoi local, timezone-naive) — convert to UTC."""
    df = pd.read_csv(csv_path, parse_dates=["datetime"])
    df["time_utc"] = df["datetime"] - pd.Timedelta(hours=7)
    return df[["time_utc", "aod_550"]].sort_values("time_utc").reset_index(drop=True)

aeronet = load_aeronet(AERONET_CSV)
aeronet = aeronet[(aeronet.time_utc >= DATE_START) & (aeronet.time_utc <= DATE_END)]
aeronet_dry = aeronet[aeronet.time_utc.dt.month.isin(DRY_MONTHS)].reset_index(drop=True)
print(f"AERONET rows in dry season: {len(aeronet_dry):,}")
aeronet_dry.head()


In [ ]:
def load_merra2_at_point(root, lat, lon, start, end, var="TOTEXTTAU"):
    """
    MERRA-2 is one .nc4 per day. We open each, slice the nearest cell, and
    concat — far cheaper than open_mfdataset over ~500 daily files.
    Timestamps in the file are centred on HH:30 (hourly time-averaged).
    """
    files = sorted(glob.glob(f"{root}/*/*/MERRA2_*.tavg1_2d_aer_Nx.*_vnm.nc4"))
    sdate, edate = start.replace("-", ""), end.replace("-", "")
    def fdate(p):
        return os.path.basename(p).split(".")[-2].split("_")[0]
    files = [f for f in files if sdate <= fdate(f) <= edate]
    print(f"Loading {len(files)} MERRA-2 daily files…")

    daily = []
    for f in files:
        with xr.open_dataset(f) as ds:
            daily.append(ds[var].sel(lat=lat, lon=lon, method="nearest").load())
    da = xr.concat(daily, dim="time").sortby("time")
    cell_lat = float(da.lat); cell_lon = float(da.lon)
    print(f"Nearest MERRA-2 cell: ({cell_lat:.3f}, {cell_lon:.3f}) "
          f"vs station ({lat:.3f}, {lon:.3f})")
    df = da.to_dataframe(name="merra2_aod").reset_index()
    return df.rename(columns={"time": "time_utc"})[["time_utc", "merra2_aod"]]

m2 = load_merra2_at_point(MERRA2_ROOT, LAT, LON, DATE_START, DATE_END)
m2 = m2[(m2.time_utc >= DATE_START) & (m2.time_utc <= DATE_END)]
m2_dry = m2[m2.time_utc.dt.month.isin(DRY_MONTHS)].reset_index(drop=True)
print(f"MERRA-2 rows in dry season: {len(m2_dry):,}  (hourly, centred HH:30)")
m2_dry.head()


## Decision: how to pair sparse AERONET with hourly MERRA-2?

MERRA-2 is **time-averaged** over each hour, with timestamps at HH:30 (the
centre of the averaging interval). The natural pairing window is therefore
**±30 minutes** — i.e. the actual averaging interval.

As in the CAMS notebook, we want **one independent pair per model step** to
keep stats unbiased toward dense observation windows. Implement the same
`pair_observations` strategy (average AERONET into each model bucket) below.


In [ ]:
def pair_observations(aeronet_df, model_df, tolerance_minutes=30):
    """Strategy A: average AERONET into each model timestep's window."""
    tol = pd.Timedelta(minutes=tolerance_minutes)
    model_col = next(c for c in model_df.columns if c != "time_utc")
    aer = aeronet_df.sort_values("time_utc").reset_index(drop=True)
    mod = model_df.sort_values("time_utc").reset_index(drop=True)

    # 1. attach each AERONET point to its nearest model timestamp (≤ tol).
    matched = pd.merge_asof(
        aer,
        mod[["time_utc"]].rename(columns={"time_utc": "model_time"}),
        left_on="time_utc", right_on="model_time",
        direction="nearest", tolerance=tol,
    ).dropna(subset=["model_time"])

    # 2. average AERONET inside each model bucket.
    agg = (matched.groupby("model_time")
                  .agg(aeronet_aod=("aod_550", "mean"),
                       n_aeronet=("aod_550", "size"))
                  .reset_index()
                  .rename(columns={"model_time": "time_utc"}))

    # 3. inner-join model AOD back on.
    out = agg.merge(mod, on="time_utc", how="inner")
    return out.rename(columns={model_col: "model_aod"})[
        ["time_utc", "aeronet_aod", "model_aod", "n_aeronet"]
    ]


pairs = pair_observations(aeronet_dry, m2_dry, tolerance_minutes=30)
print(f"Paired observations: {len(pairs):,}")
pairs.head()


In [ ]:
def summarise(df):
    a = df["aeronet_aod"].to_numpy()
    m = df["model_aod"].to_numpy()
    haze = a > 1.0
    return pd.Series({
        "n":                  len(a),
        "bias (MERRA2-AER)":  float(np.mean(m - a)),
        "RMSE":               float(np.sqrt(np.mean((m - a) ** 2))),
        "r":                  float(np.corrcoef(a, m)[0, 1]),
        "n_haze (AER>1)":     int(haze.sum()),
        "<AER>|haze":         float(a[haze].mean()) if haze.any() else np.nan,
        "<MERRA2>|haze":      float(m[haze].mean()) if haze.any() else np.nan,
        "bias|haze":          float(np.mean(m[haze] - a[haze])) if haze.any() else np.nan,
    })

summarise(pairs)


In [ ]:
seasons = [("2022-11-01", "2023-04-01"),
           ("2023-11-01", "2024-04-01"),
            ("2024-11-01", "2025-04-01"),
           ("2025-11-01", "2026-04-01")]

fig, axes = plt.subplots(len(seasons), 1, figsize=(13, 3.0 * len(seasons)),
                         sharey=True)
if len(seasons) == 1:
    axes = [axes]

for ax, (s, e) in zip(axes, seasons):
    sub = pairs[(pairs.time_utc >= s) & (pairs.time_utc < e)]
    if sub.empty:
        ax.set_title(f"{s[:7]} — {e[:7]}  (no data)")
        continue
    ax.plot(sub.time_utc, sub.aeronet_aod, "o", ms=2.5, color="black",
            alpha=0.6, label="AERONET (mean in ±30 min)")
    ax.plot(sub.time_utc, sub.model_aod, "-", color="tab:blue", lw=1.3,
            alpha=0.85, label="MERRA-2 (nearest cell)")
    ax.set_ylabel("AOD 550")
    ax.set_title(f"Dry season {s[:7]} → {e[:7]}")
    ax.grid(alpha=0.3)
    ax.legend(loc="upper right", fontsize=9)
axes[-1].set_xlabel("UTC")
fig.suptitle("MERRA-2 vs AERONET @ Nghia Do — NE-monsoon dry season", y=1.00)
fig.tight_layout()


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.scatter(pairs.aeronet_aod, pairs.model_aod, s=14, alpha=0.45, edgecolor="none")
m = max(pairs.aeronet_aod.max(), pairs.model_aod.max()) * 1.05
ax.plot([0, m], [0, m], "k--", lw=1, label="1:1")
haze = pairs[pairs.aeronet_aod > 1.0]
ax.scatter(haze.aeronet_aod, haze.model_aod, s=30, facecolor="none",
           edgecolor="tab:red", lw=1.2, label=f"haze (AER>1, n={len(haze)})")
ax.set_xlim(0, m); ax.set_ylim(0, m)
ax.set_xlabel("AERONET AOD 550"); ax.set_ylabel("MERRA-2 AOD 550")
s = summarise(pairs)
ax.set_title(f"n={int(s['n'])}  bias={s['bias (MERRA2-AER)']:+.3f}  "
             f"RMSE={s['RMSE']:.3f}  r={s['r']:.3f}")
ax.grid(alpha=0.3); ax.legend(loc="upper left")
fig.tight_layout()
